# 03: Investor Transactions — Data Cleaning & Pre-processing

**Day 2 | Part 2 — Bluestock Mutual Fund Analytics**

Pipeline steps:
1. Load raw `08_investor_transactions.csv`
2. Parse & standardise dates → `datetime64`
3. Standardise `transaction_type` values (SIP / Lumpsum / Redemption)
4. Validate `amount_inr` > 0
5. Validate & standardise `kyc_status` values (Verified / Pending / Rejected)
6. Remove duplicates
7. Save `data/processed/clean_transactions.csv`
8. Verification summary

In [1]:
import pandas as pd
import re
from pathlib import Path

RAW_PATH      = Path('../data/raw/08_investor_transactions.csv')
PROCESSED_DIR = Path('../data/processed')
OUTPUT_PATH   = PROCESSED_DIR / 'clean_transactions.csv'

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

## Step 1 — Load Raw Data

In [2]:
df_raw = pd.read_csv(RAW_PATH)

print(f"Raw shape   : {df_raw.shape}")
print(f"Columns     : {df_raw.columns.tolist()}")
print(f"\nDtypes:\n{df_raw.dtypes}\n")
df_raw.head(3)

Raw shape   : (32778, 13)
Columns     : ['investor_id', 'transaction_date', 'amfi_code', 'transaction_type', 'amount_inr', 'state', 'city', 'city_tier', 'age_group', 'gender', 'annual_income_lakh', 'payment_mode', 'kyc_status']

Dtypes:
investor_id            object
transaction_date       object
amfi_code               int64
transaction_type       object
amount_inr              int64
state                  object
city                   object
city_tier              object
age_group              object
gender                 object
annual_income_lakh    float64
payment_mode           object
kyc_status             object
dtype: object



,investor_id,transaction_date,amfi_code,transaction_type,amount_inr,state,city,city_tier,age_group,gender,annual_income_lakh,payment_mode,kyc_status
0,INV003054,2024-01-01,119092,SIP,1834,Telangana,Hyderabad,T30,56+,Female,77.1,UPI,Verified
1,INV002952,2024-01-01,148567,Redemption,392882,Punjab,Amritsar,B30,18-25,Male,7.1,Cheque,Verified
2,INV003420,2024-01-01,118636,SIP,912,Haryana,Faridabad,B30,36-45,Male,47.2,Mandate,Verified


## Step 2 — Parse & Standardise Dates

In [3]:
df = df_raw.copy()

# Convert object → datetime64; errors='coerce' turns unparseable values into NaT
df['transaction_date'] = pd.to_datetime(df['transaction_date'], errors='coerce')

unparseable = df['transaction_date'].isna().sum()
print(f"Unparseable dates (NaT): {unparseable}")

if unparseable:
    # Log bad rows before dropping
    bad_dates = df_raw[df['transaction_date'].isna()]
    print(f"Bad date rows:\n{bad_dates}")
    df.dropna(subset=['transaction_date'], inplace=True)
    df.reset_index(drop=True, inplace=True)
    print(f"Rows after dropping NaT dates: {len(df)}")

print(f"Date range  : {df['transaction_date'].min().date()}  →  {df['transaction_date'].max().date()}")
print(f"date dtype  : {df['transaction_date'].dtype}")

Unparseable dates (NaT): 0
Date range  : 2024-01-01  →  2025-05-30
date dtype  : datetime64[ns]


## Step 3 — Standardise transaction_type

Allowed values: **SIP**, **Lumpsum**, **Redemption**

In [4]:
VALID_TXN_TYPES = {'SIP', 'Lumpsum', 'Redemption'}

# Normalise: strip whitespace, title-case
df['transaction_type'] = df['transaction_type'].str.strip().str.title()

# Further canonical mapping (handles 'LUMPSUM', 'lump sum', 'redemption' etc.)
txn_map = {
    'Sip'        : 'SIP',
    'Lump Sum'   : 'Lumpsum',
    'Lumpsum'    : 'Lumpsum',
    'Redemption' : 'Redemption',
    'Redeem'     : 'Redemption',
}
df['transaction_type'] = df['transaction_type'].replace(txn_map)

print(f"transaction_type value counts:")
print(df['transaction_type'].value_counts())

invalid_types = df[~df['transaction_type'].isin(VALID_TXN_TYPES)]
print(f"\nInvalid transaction_type rows: {len(invalid_types)}")
if len(invalid_types) > 0:
    print(invalid_types['transaction_type'].unique())
    df = df[df['transaction_type'].isin(VALID_TXN_TYPES)].reset_index(drop=True)
    print(f"Rows after removing invalid types: {len(df)}")
else:
    print("All transaction_type values are valid.")

transaction_type value counts:
transaction_type
SIP           19716
Lumpsum        8095
Redemption     4967
Name: count, dtype: int64

Invalid transaction_type rows: 0
All transaction_type values are valid.


## Step 4 — Validate amount_inr > 0

In [5]:
invalid_amount = df[df['amount_inr'] <= 0]
print(f"Invalid amount_inr rows (amount <= 0): {len(invalid_amount)}")

if len(invalid_amount) > 0:
    print(f"\nInvalid rows:")
    print(invalid_amount[['investor_id','transaction_date','transaction_type','amount_inr']].to_string())
    # Save flagged rows for audit
    invalid_amount.to_csv(PROCESSED_DIR / 'flagged_invalid_amount.csv', index=False)
    df = df[df['amount_inr'] > 0].reset_index(drop=True)
    print(f"Rows after removing invalid amounts: {len(df)}")
else:
    print("All amount_inr values are positive — no anomalies found.")

print(f"\namount_inr stats:")
print(df['amount_inr'].describe().round(2))

Invalid amount_inr rows (amount <= 0): 0
All amount_inr values are positive — no anomalies found.

amount_inr stats:
count     32778.00
mean     107437.32
std      150415.91
min         400.00
25%        3153.00
50%       17782.50
75%      189324.25
max      597498.00
Name: amount_inr, dtype: float64


## Step 5 — Validate & Standardise kyc_status

Allowed values: **Verified**, **Pending**, **Rejected**

In [6]:
VALID_KYC = {'Verified', 'Pending', 'Rejected'}

# Normalise: strip, title-case
df['kyc_status'] = df['kyc_status'].str.strip().str.title()

print(f"kyc_status value counts:")
print(df['kyc_status'].value_counts())

invalid_kyc = df[~df['kyc_status'].isin(VALID_KYC)]
print(f"\nInvalid kyc_status rows: {len(invalid_kyc)}")
if len(invalid_kyc) > 0:
    print(f"Unexpected KYC values: {invalid_kyc['kyc_status'].unique()}")

kyc_status value counts:
kyc_status
Verified    30146
Pending      2632
Name: count, dtype: int64

Invalid kyc_status rows: 0


## Step 6 — Remove Duplicates

In [7]:
dups = df.duplicated().sum()
print(f"Duplicate rows: {dups}")

if dups:
    df.drop_duplicates(inplace=True)
    df.reset_index(drop=True, inplace=True)
    print(f"Shape after dedup: {df.shape}")
else:
    print("No duplicates found — nothing to drop.")

Duplicate rows: 0
No duplicates found — nothing to drop.


## Step 7 — Save Clean Dataset

In [8]:
# Final column order — match raw schema
df_clean = df[[
    'investor_id', 'transaction_date', 'amfi_code', 'transaction_type',
    'amount_inr', 'state', 'city', 'city_tier', 'age_group', 'gender',
    'annual_income_lakh', 'payment_mode', 'kyc_status'
]].copy()

# Write ISO date format
df_clean['transaction_date'] = df_clean['transaction_date'].dt.strftime('%Y-%m-%d')

df_clean.to_csv(OUTPUT_PATH, index=False)
print(f"Saved → {OUTPUT_PATH}")
print(f"Final row count: {len(df_clean):,}")

Saved → ../data/processed/clean_transactions.csv
Final row count: 32,778


## Step 8 — Verification Summary

In [9]:
df_verify = pd.read_csv(OUTPUT_PATH, parse_dates=['transaction_date'])

print("=" * 55)
print("  CLEAN TRANSACTIONS — VERIFICATION REPORT")
print("=" * 55)

print("\n[1] Shape           :", df_verify.shape)

print("\n[2] Null counts")
print(df_verify.isnull().sum())

print("\n[3] Duplicate rows  :", df_verify.duplicated().sum())
assert df_verify.duplicated().sum() == 0, "ERROR: duplicates remain!"

print("[4] Invalid amount  :", (df_verify['amount_inr'] <= 0).sum())
assert (df_verify['amount_inr'] <= 0).sum() == 0, "ERROR: invalid amounts remain!"

print("\n[5] transaction_type values:")
print(df_verify['transaction_type'].value_counts())
assert set(df_verify['transaction_type'].unique()).issubset({'SIP', 'Lumpsum', 'Redemption'}), \
    "ERROR: invalid transaction types!"

print("\n[6] kyc_status values:")
print(df_verify['kyc_status'].value_counts())
assert set(df_verify['kyc_status'].unique()).issubset({'Verified', 'Pending', 'Rejected'}), \
    "ERROR: invalid KYC status values!"

print("\n[7] Date range      :", df_verify['transaction_date'].min().date(), "→", df_verify['transaction_date'].max().date())

print("\n" + "=" * 55)
print("  ALL CHECKS PASSED — clean_transactions.csv is ready")
print("=" * 55)

  CLEAN TRANSACTIONS — VERIFICATION REPORT

[1] Shape           : (32778, 13)

[2] Null counts
investor_id           0
transaction_date      0
amfi_code             0
transaction_type      0
amount_inr            0
state                 0
city                  0
city_tier             0
age_group             0
gender                0
annual_income_lakh    0
payment_mode          0
kyc_status            0
dtype: int64



[3] Duplicate rows  : 0


[4] Invalid amount  : 0

[5] transaction_type values:
transaction_type
SIP           19716
Lumpsum        8095
Redemption     4967
Name: count, dtype: int64

[6] kyc_status values:
kyc_status
Verified    30146
Pending      2632
Name: count, dtype: int64

[7] Date range      : 2024-01-01 → 2025-05-30

  ALL CHECKS PASSED — clean_transactions.csv is ready
